In [ ]:
from calitp_data_analysis.gcs_pandas import GCSPandas
from functools import cache

import pandas as pd
import geopandas as gpd
from IPython.core.display import HTML

In [ ]:
@cache
def gcs_pandas():
    return GCSPandas()

In [ ]:
%env REQUESTS_CA_BUNDLE=C:\Users\s163107\Documents\CTROOTCA01.cer

In [ ]:
culver_tsp_extract = gcs_pandas().read_csv(
    "gs://calitp-analytics-data/data-analyses/tsp-analysis/culver_city_tsp_validation/VehicleState0007140250506_055240.txt",
    skiprows=lambda x: x == 1,
)

## Very basic investigation

In [ ]:
culver_tsp_extract.head()

In [ ]:
list(culver_tsp_extract.columns)

In [ ]:
# See routes present
(culver_tsp_extract.ROUTE_ID.astype(str) + "_" + culver_tsp_extract.TRIP_KEY.astype(str)).value_counts().sort_index()

In [ ]:
# Process time data into python format
culver_tsp_extract_sorted = culver_tsp_extract.sort_values(["TRIP_KEY", "EVENT_TIME"], ascending=True)
culver_tsp_extract_sorted["event_time_datetime"] = pd.to_datetime(
    culver_tsp_extract_sorted["EVENT_TIME"].astype(str).fillna("000000000000.0"),
    format=r"%y%m%d%H%M%S.0",
)
culver_tsp_extract_sorted[["EVENT_TIME", "event_time_datetime"]].head()
culver_tsp_extract_sorted["time_difference"] = (
    culver_tsp_extract_sorted.groupby("TRIP_KEY")["event_time_datetime"].diff().dt.total_seconds()
)
culver_tsp_extract_sorted["time_difference"].describe()

In [ ]:
# Select a representative trip
trip_1181 = culver_tsp_extract_sorted.loc[culver_tsp_extract_sorted.TRIP_KEY == 1181]

### Mapping

In [ ]:
# Remove long gaps so the scale works - these are mostly at the start and end of trips
gdf_trip_1181 = gpd.GeoDataFrame(
    trip_1181, geometry=gpd.points_from_xy(trip_1181.LONGITUDE, trip_1181.LATITUDE), crs="EPSG:4326"
).loc[trip_1181.time_difference < 30]
gdf_trip_1181[["geometry", "time_difference"]].explore("time_difference")

### Frequency Validation

In [ ]:
# Histogram of frequencies for example trip
trip_1181["time_difference"].hist(bins=50)

In [ ]:
# Removing outliers
trip_1181.loc[trip_1181.time_difference < 30, "time_difference"].hist(bins=30)

### Histograms for all trips

In [ ]:
culver_tsp_extract_sorted["time_difference"].hist(bins=30)

In [ ]:
# Removing outliers
culver_tsp_extract_sorted.loc[culver_tsp_extract_sorted.time_difference < 30, "time_difference"].hist(bins=30)